### Import Library & Load dataset

In [1]:
from datasets import load_dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


### Inspecting label names

In [2]:
label_names = dataset["train"].features["label"].names
print("Label Names:", label_names)


Label Names: ['negative', 'neutral', 'positive']


In [3]:
dataset["train"][0]


{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"',
 'label': 2}

In [4]:
import numpy as np

train_labels = dataset["train"]["label"]
unique, counts = np.unique(train_labels, return_counts=True)

for label, count in zip(unique, counts):
    print(f"{label_names[label]}: {count}")

negative: 7093
neutral: 20673
positive: 17849


### Import necessary libraries

In [5]:
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader


### Load the device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


### Load Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### Tokenization Function

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


### Apply Tokenization

In [9]:
tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_val = dataset["validation"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)


Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

### Remove Unnecessary Columns

In [10]:
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_val = tokenized_val.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])


### Set Torch Format

In [11]:
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")


### Create DataLoaders

In [12]:
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    tokenized_val,
    batch_size=16,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    tokenized_test,
    batch_size=16,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders ready")


DataLoaders ready


### Import Model

In [13]:
from transformers import AutoModelForSequenceClassification


### Load DistilRoBERTa

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

model.to(device)

print("Model loaded successfully")


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


### Verify Output Shape

In [15]:
batch = next(iter(train_dataloader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print("Logits shape:", outputs.logits.shape)


Logits shape: torch.Size([16, 3])


### Import Training Utilities

In [16]:
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


### Define Loss Function

In [17]:
criterion = nn.CrossEntropyLoss()


### Optimizer

In [18]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)


### Scheduler

In [19]:
epochs = 2

total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print("Training setup ready")


Training setup ready


### Training

In [20]:
from tqdm import tqdm

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # -------- TRAIN --------
    model.train()
    total_train_loss = 0
    correct_train = 0
    total_train = 0

    train_progress = tqdm(train_dataloader)

    for batch in train_progress:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        total_train_loss += loss.item()

        # Accuracy tracking
        preds = torch.argmax(logits, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_progress.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_dataloader)
    train_accuracy = correct_train / total_train

    # -------- VALIDATION --------
    model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_accuracy = correct_val / total_val

    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")



Epoch 1/2


100%|██████████| 2851/2851 [17:35<00:00,  2.70it/s, loss=0.499]


Training Loss: 0.6722
Training Accuracy: 0.6947
Validation Loss: 0.6675
Validation Accuracy: 0.7115

Epoch 2/2


100%|██████████| 2851/2851 [17:40<00:00,  2.69it/s, loss=0.427]


Training Loss: 0.4920
Training Accuracy: 0.7886
Validation Loss: 0.6149
Validation Accuracy: 0.7540


### Evaluation

In [22]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

micro_f1 = f1_score(all_labels, all_preds, average="micro")
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("Test Micro F1:", micro_f1)
print("Test Macro F1:", macro_f1)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


Test Micro F1: 0.705958971019212
Test Macro F1: 0.7066680791751957

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.81      0.74      3972
           1       0.76      0.61      0.68      5937
           2       0.64      0.77      0.70      2375

    accuracy                           0.71     12284
   macro avg       0.70      0.73      0.71     12284
weighted avg       0.72      0.71      0.70     12284



In [23]:
import os

SAVE_PATH = "models/sentiment_model_roberta_base"

os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Sentiment model saved successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sentiment model saved successfully!


In [24]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [25]:
import os

SAVE_PATH = "/content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base"

os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to Google Drive successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive successfully!


In [26]:
!zip -r sentiment_model_roberta_base.zip /content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base


  adding: content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base/ (stored 0%)
  adding: content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base/config.json (deflated 52%)
  adding: content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base/model.safetensors (deflated 9%)
  adding: content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base/tokenizer_config.json (deflated 50%)
  adding: content/drive/MyDrive/mindtrace_models/sentiment_model_roberta_base/tokenizer.json (deflated 82%)


In [27]:
from google.colab import files
files.download("sentiment_model_roberta_base.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Preparing to continue training

In [28]:
from transformers import get_linear_schedule_with_warmup

additional_epochs = 2

total_steps = len(train_dataloader) * additional_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

epochs = additional_epochs

print("Scheduler reset for continued training")


Scheduler reset for continued training


### Train again

In [31]:
from tqdm import tqdm
import torch

for epoch in range(epochs):
    print(f"\nContinued Epoch {epoch+1}/{epochs}")

    # -------- TRAIN --------
    model.train()
    total_train_loss = 0
    correct_train = 0
    total_train = 0

    train_progress = tqdm(train_dataloader)

    for batch in train_progress:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        total_train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        train_progress.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_dataloader)
    train_accuracy = correct_train / total_train

    # -------- VALIDATION --------
    model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_accuracy = correct_val / total_val

    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")



Continued Epoch 1/2


100%|██████████| 2851/2851 [17:41<00:00,  2.69it/s, loss=0.43]


Training Loss: 0.4777
Training Accuracy: 0.7982
Validation Loss: 0.6396
Validation Accuracy: 0.7425

Continued Epoch 2/2


100%|██████████| 2851/2851 [17:42<00:00,  2.68it/s, loss=0.135]


Training Loss: 0.3398
Training Accuracy: 0.8654
Validation Loss: 0.7581
Validation Accuracy: 0.7430


Validation loss is spiking continously from epoch 2. Clearly overfiting, Training Accuracy improving but in parallel Validation loss is increasing it means model is good at memorizing trained daa but failing to work on unseen data